In [1]:
from operator import add
from typing import Annotated, TypedDict
from langgraph.constants import START, END
from langgraph.graph import StateGraph

#定义State
class ParallelState(TypedDict):
    nodes : Annotated[list[str] , add]

#定义图
builder_graph = StateGraph(ParallelState)

#定义node
def a(state:ParallelState):
    print(f"传入a的值:{state["nodes"]}")
    state["nodes"] = ["a"]
    return state
def b(state:ParallelState):
    print(f"传入b的值:{state["nodes"]}")
    state["nodes"] = ["b"]
    return state
def c(state:ParallelState):
    print(f"传入c的值:{state["nodes"]}")
    state["nodes"] = ["c"]
    return state
def d(state:ParallelState):
    print(f"传入d的值:{state["nodes"]}")
    state["nodes"] = ["d"]
    return state

builder_graph.add_node("a",a)
builder_graph.add_node("b",b)
builder_graph.add_node("c",c)
builder_graph.add_node("d",d)

#定义边
builder_graph.add_edge(START,"a")
builder_graph.add_edge("a","b")
builder_graph.add_edge("a","c")
builder_graph.add_edge("c","d")
builder_graph.add_edge("b","d")
builder_graph.add_edge("d",END)

#编译图
graph = builder_graph.compile()

#执行图
graph.invoke({"nodes": []})

传入a的值:[]
传入b的值:['a']
传入c的值:['a']
传入d的值:['a', 'b', 'c']


{'nodes': ['a', 'b', 'c', 'd']}

In [2]:
#################################### 生成并保存图 #########################################

# 1. 查看图的 Mermaid 源码（纯本地，无需网络）
print(graph.get_graph().draw_mermaid())

# 2. 渲染为 PNG 并保存（需要联网，走 mermaid.ink 渲染）
png_data = graph.get_graph().draw_mermaid_png()
with open("edge_graph.png", "wb") as f:
    f.write(png_data)

print("图表已成功保存为 edge_graph.png")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	a(a)
	b(b)
	c(c)
	d(d)
	__end__([<p>__end__</p>]):::last
	__start__ --> a;
	a --> b;
	a --> c;
	b --> d;
	c --> d;
	d --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

图表已成功保存为 edge_graph.png


In [2]:
from typing import TypedDict, Literal
from langgraph.constants import START, END
from langgraph.graph import StateGraph


# =================1. 定义State数据结构================
class RouteState(TypedDict):
    score: int
    res: str

# =================2. Node: 定义处理逻辑=================
def scorer(state: RouteState):
    score = int(input("请输入分数:"))
    state["score"] = score
    return state

def session(state: RouteState):
    state["res"] =  "通过！"
    return state

def fail(state: RouteState):
    state["res"] =  "不通过！"
    return state

# =================3. 定义路由节点，决定下个Node=================
def router(state: RouteState) -> Literal["分数大于60", "分数小于60"]:
    """根据score决定下一节点"""
    if state["score"] >= 60:
        return "分数大于60"
    return "分数小于60"

# ===================4. 创建Graph===================

builder = StateGraph(RouteState)                # 1.创建Graph
                                                # 2.添加节点
builder.add_node("分数节点", scorer)                  # 打分节点
builder.add_node("分数大于60", session)                 # 失败节点
builder.add_node("分数小于60", fail)                 # 成功节点
                                                # 3.添加Edge
builder.add_edge(START, "分数节点")                   # start -> 分数节点
builder.add_conditional_edges("分数节点", router)     # 分数节点 -> router （条件边）
builder.add_edge("分数大于60", END)                       # 分数大于60 -> end
builder.add_edge("分数小于60", END)                       # 分数小于60 -> end
condition_graph=builder.compile()


# =================5.调用Graph，传入默认State值=================
result = condition_graph.invoke({})
print(result)


{'score': 60, 'res': '通过！'}


In [3]:
#################################### 生成并保存条件边图 #########################################

# 1. 查看图的 Mermaid 源码（纯本地，无需网络）
print(condition_graph.get_graph().draw_mermaid())

# 2. 渲染为 PNG 并保存（需要联网，走 mermaid.ink 渲染）
png_data = condition_graph.get_graph().draw_mermaid_png()
with open("condition_graph.png", "wb") as f:
    f.write(png_data)

print("图表已成功保存为 condition_graph.png")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	\5206\6570\8282\70b9(分数节点)
	\5206\6570\5927\4e8e60(分数大于60)
	\5206\6570\5c0f\4e8e60(分数小于60)
	__end__([<p>__end__</p>]):::last
	__start__ --> \5206\6570\8282\70b9;
	\5206\6570\8282\70b9 -.-> \5206\6570\5927\4e8e60;
	\5206\6570\8282\70b9 -.-> \5206\6570\5c0f\4e8e60;
	\5206\6570\5927\4e8e60 --> __end__;
	\5206\6570\5c0f\4e8e60 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

图表已成功保存为 condition_graph.png
